# Mecanismos de Atenção

## 1 Atenção causal e bidirecional

Durante a aula você aprendeu sobre o mencanismo de atenção usado nos modelos de linguagem atuais. Criamos a classe `SelfAttention_v2` com atenção bidirecional, onde cada token pode dar atenção tanto para os tokens que vem antes na sentença, como pode dar atenção para os tokens subsequentes. Em seguida, adicionamos causalidade na atenção e criamos a classe `CausalAttention`, que faz com que os tokens dêem atenção apenas para os tokens passados, comumente usada nos grandes modelos de linguagem generativos.
<br><br>
Porém, modelos como o T5 usam a arquitetura enconder-decoder, que mistura os dois tipos de atenção: bidirecional (nos enconders) e causal (nos decoders). Tendo isso em mente, faça o seguinte:

- Baseando-se na classe `CausalAttention`, crie a classe `SelfAttention_v3` que poderá se comportar tanto com atenção bidirecional quanto com atenção causal. Para isso, adicione o parâmetro booleano `is_causal` no construtor da classe, que aplicará o filtro causal na matriz de atenção caso o parâmetro seja verdadeiro e bidirecional caso seja falso. Altere também o retorno do método forward para retornar tanto o `context_vec`quanto o `attn_weights`.

- Em seguida passe o input definido abaixo pela pela nova classe `SelfAttention_v3` usando a arquitetura bidirecional e também a arquitetura causal e imprima o resultado dos vetores de contexto e dos pesos de atenção nas duas situações para comparar a diferença entre eles.

In [51]:
import torch
import torch.nn as nn

inputs = torch.tensor(
  [[[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]]] # step     (x^6)
)

d_in, d_out = 3, 2
context_length=6
dropout=0.0

In [52]:
class SelfAttention_v3(nn.Module):
    def __init__(self, d_in, d_out, tamanho_contexto, dropout=0, qkvbias=False, is_causal=True): # COMPLETE OS PARÂMETROS # passo o "is_causal" pro trem
        super().__init__()

        self.d_out = d_out
        self.is_causal = is_causal
        # SEU CÓDIGO
        self.W_query = nn.Linear(d_in,d_out,bias=qkvbias) # linear para Wq
        self.W_key = nn.Linear(d_in,d_out,bias=qkvbias) # linear para Wk
        self.W_value = nn.Linear(d_in,d_out,bias=qkvbias) # linear para Wv
        self.dropout = nn.Dropout(dropout) # Dropout parece ser usado pelo enunciado (embora =0) vou deixar aqui
        if is_causal: # verifico se é causal como no exercício é pedido
            self.register_buffer('mask', torch.triu(torch.ones(tamanho_contexto,tamanho_contexto), diagonal=1)) # Exemplo do livro usava e fui pesquisar motivos da boa prática
            # self.mask = torch.triu(torch.ones(tamanho_contexto, tamanho_contexto), diagonal=1) # aparentemente da problema se rodar na GPU e não será salvo junto do modelo no dicionário do torch.save caso use...


    def forward(self, x): # COMPLETE OS PARÂMETROS

        # SEU CÓDIGO
        batch, num_tokens, _ = x.shape # aqui eu desmonto a tupla do x = (batch, tokens, d_in). Como não vou usar o d_in e já passo ele no init tirei com "_"


        Q_matrix = self.W_query(x)
        K_matrix = self.W_key(x)
        V_matrix = self.W_value(x)

        # Como isso é algo iterativo/temporario não preciso guardar na memoria se não da xabu dps
        # self.Q = self.W_query(x)
        # self.K = self.W_key(x)
        # self.V = self.W_value(x)

        # self.attention_scores_omegazinho = Q_matrix @ K_matrix.T

        # Aqui, como vou trampa com batches e a dim desses trem tudo de Q,K,V é (batches, tokens/palavras/embeddings , d_out) não posso correr o risco do Transpor trocar a dimensão errada
        # Por isso vou usar o .transpose(1,2) (vou transpor a dim 1 com a 2)
        # Literalmente igual o exemplo que estudei
        attention_scores_omegazinho = Q_matrix @ K_matrix.transpose(1,2) # não uso self.attentionscores pq tbm é algo repetitivo e temporário (poderia dar xabu de memoria...)
        if self.is_causal:
            mascara = self.mask # acaba existindo depois do self.register_buffer('mask')
            # attention_scores_omegazinho = torch.masked_fill(mascara.bool(), -torch.inf)
            # attention_scores_omegazinho = attention_scores_omegazinho.masked_fill(mascara.bool()[:num_tokens,:num_tokens], -torch.inf) # coloco o limite do número de tokens para caso num_tokens<tamanho_contexto

            # Como masked_fill aparece sem o "_" no final ele é out-place (pelo que li na documentação), logo precisaria sobrescrever dessa forma que fiz.
            # Da forma do exemplo fica melhor pq é in-place e economiza memória
            attention_scores_omegazinho.masked_fill_(mascara.bool()[:num_tokens,:num_tokens], -torch.inf) # aqui é inplace agr :)

            # Agr tenho minha matriz com vários -inf acima da diagonal principal (sem contar ela é claro)
            # caso eu aplique o softmax todos os e^-inf vão sumir e terei uma normalização bonitinha direto do processo

        attention_weights = torch.softmax(attention_scores_omegazinho/self.d_out**0.5, dim=-1) #é aquele trem de dividir por raiz da dimensão. Como quero que a soma de cada elemento numa linha (logo percorro colunas de uma linha) totalizem 1 uso "dim=-1" para isso.
        # o livro usa keys.shape[-1], mas pelo que entendi ao estudar é a mesma coisa que o d_out então da na mesma.

        attention_weights = self.dropout(attention_weights)

        vetor_contexto = attention_weights @ V_matrix
        return vetor_contexto, attention_weights # exercicio me pediu para colocar o trem do "attention_weights" tbm


In [53]:
# PASSE O INPUT PELA SelfAttention_v3 COM O PARÂMETRO is_causal=True
# E IMPRIMA A MATRIZ DE ATENÇÃO RETORNADA

# passando uma seed (a mesma do exemplo vai que o professor usou ela e ajuda ele na correção)
torch.manual_seed(123)

print(inputs.shape) # já está 3D, logo num preciso de fazer aquele trem de stack e batch

attention = SelfAttention_v3(d_in,d_out,context_length, dropout, qkvbias=False, is_causal=True) # crio o attention


vetor_contexto, pesos_attention = attention(inputs)

print("=" * 60)
print(f"VETORES DE CONTEXTO (Shape: {vetor_contexto.shape})")
print("=" * 60)
print(vetor_contexto)
print("\n" + "=" * 60)
print(f"MATRIZ DE PESOS DE ATENÇÃO CAUSAL (is_causal=True) (Shape: {pesos_attention.shape})")
print("=" * 60)
print(pesos_attention)


torch.Size([1, 6, 3])
VETORES DE CONTEXTO (Shape: torch.Size([1, 6, 2]))
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)

MATRIZ DE PESOS DE ATENÇÃO CAUSAL (is_causal=True) (Shape: torch.Size([1, 6, 6]))
tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
         [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
         [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
         [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]]],
       grad_fn=<SoftmaxBackward0>)


In [54]:
# PASSE O INPUT PELA SelfAttention_v3 COM O PARÂMETRO is_causal=false
# E IMPRIMA A MATRIZ DE ATENÇÃO RETORNADA
torch.manual_seed(123)

print(inputs.shape) # já está 3D, logo num preciso de fazer aquele trem de stack e batch

attention = SelfAttention_v3(d_in,d_out,context_length, dropout, qkvbias=False, is_causal=False) # crio o attention


vetor_contexto, pesos_attention = attention(inputs)

print("=" * 60)
print(f"VETORES DE CONTEXTO (Shape: {vetor_contexto.shape})")
print("=" * 60)
print(vetor_contexto)
print("\n" + "=" * 60)
print(f"MATRIZ DE PESOS DE ATENÇÃO BIDIRECIONAL (is_causal=False) (Shape: {pesos_attention.shape})")
print("=" * 60)
print(pesos_attention)

torch.Size([1, 6, 3])
VETORES DE CONTEXTO (Shape: torch.Size([1, 6, 2]))
tensor([[[-0.5337, -0.1051],
         [-0.5323, -0.1080],
         [-0.5323, -0.1079],
         [-0.5297, -0.1076],
         [-0.5311, -0.1066],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)

MATRIZ DE PESOS DE ATENÇÃO BIDIRECIONAL (is_causal=False) (Shape: torch.Size([1, 6, 6]))
tensor([[[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
         [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
         [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
         [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
         [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
         [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]]],
       grad_fn=<SoftmaxBackward0>)


## 2 Multi-head Attention + Data Loading

Use o data loader visto nas aulas anteriores para tokenizar e processar o texto abaixo para a tarefa de previsão do próximo token e passe o primeiro batch de dados pela camada de multi-head attention visto na aula passada e imprima o shape da saída. Lembre-se que os tokens gerados pelo data loader devem passar pela camada de token embeddings e position embeddings (token_embedding + position_embedding) antes de passar pela atenção. Essas camadas também já foram definidas abaixo.

In [55]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


time_machine_text = "The Time Traveller (for so it will be convenient to speak of him) \
was expounding a recondite matter to us. His grey eyes shone and \
twinkled, and his usually pale face was flushed and animated. The \
fire burned brightly, and the soft radiance of the incandescent \
lights in the lilies of silver caught the bubbles that flashed and \
passed in our glasses. Our chairs, being his patents, embraced and \
caressed us rather than submitted to be sat upon, and there was that \
luxurious after-dinner atmosphere when thought roams gracefully \
free of the trammels of precision. And he put it to us in this \
way--marking the points with a lean forefinger--as we sat and lazily \
admired his earnestness over this new paradox (as we thought it) \
and his fecundity."


vocab_size = 50257
emb_dim = 256
context_length = 1024


token_embedding_layer = nn.Embedding(vocab_size, emb_dim)
print(token_embedding_layer)
pos_embedding_layer = nn.Embedding(context_length, emb_dim)
print(pos_embedding_layer)

Embedding(50257, 256)
Embedding(1024, 256)


In [56]:
# DEFINA A CLASSE DE DATASET, O DATA LOADER E A CLASSE DE MULTI HEAD ATTENTION
class DatasetGPT_aula(Dataset):
    def __init__(self, texto, tokenizador, tamanho_janela, stride):
        self.input_tokens_ids=[]
        self.target_tokens_ids=[]

        dados_tokenizados_id = tokenizador.encode(texto)

        for i in range(0, len(dados_tokenizados_id)-tamanho_janela , stride):
            self.input_tokens_ids.append(torch.tensor(dados_tokenizados_id[i:i+tamanho_janela]))
            self.target_tokens_ids.append(torch.tensor(dados_tokenizados_id[i+1:i+tamanho_janela+1]))

    def __len__(self):
        return len(self.input_tokens_ids)

    def __getitem__(self, idx):
        return self.input_tokens_ids[idx], self.target_tokens_ids[idx]

def criar_data_loader(texto, batch_size=4, tamanho_janela=2, stride=1 ,shuffle=True, drop_last=True, num_workers=0):

    tokenizador_data_load = tiktoken.get_encoding("gpt2")

    dataset = DatasetGPT_aula(texto, tokenizador_data_load, tamanho_janela, stride)

    dataloader = DataLoader(
        dataset, #objeto da classe criada
        batch_size=batch_size, #tamanho do lote. Numero de amostras agrupados por tensor.
        shuffle=shuffle, # embaralhar os dados
        drop_last=drop_last, # se o numero de dados não for perfeitamente divisivel pelo tamanho de lotes ele descarta o restinho (descarta o lote incompleto)
        num_workers=num_workers # processamentos em paralelo processador
    )
    return dataloader


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, tamanho_contexto, dropout, num_heads, qkvbias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out tem que ser divisivel por num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in,d_out,bias=qkvbias) # linear para Wq
        self.W_key = nn.Linear(d_in,d_out,bias=qkvbias) # linear para Wk
        self.W_value = nn.Linear(d_in,d_out,bias=qkvbias) # linear para Wv
        self.Wout_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout) # Dropout parece ser usado pelo enunciado (embora =0) vou deixar aqui
        self.register_buffer('mask', torch.triu(torch.ones(tamanho_contexto,tamanho_contexto), diagonal=1)) # mesma ideia
    

    def forward(self, x):
        batch, num_tokens, _ = x.shape

        Q_matrix = self.W_query(x) # (b, num_tokens, d_out)
        K_matrix = self.W_key(x) # (b, num_tokens, d_out)
        V_matrix = self.W_value(x) # (b, num_tokens, d_out)

        # uso o .view() para fatiar a dimensão do número colunas que representam meus tokens na quantidade de cabeças. Divido o vetor nessa quantidade
        Q_matrix = Q_matrix.view(batch, num_tokens, self.num_heads, self.head_dim)
        K_matrix = K_matrix.view(batch, num_tokens, self.num_heads, self.head_dim) 
        V_matrix = V_matrix.view(batch, num_tokens, self.num_heads, self.head_dim)

        # Agora reeorganizo para rodar em paralelo (pelo que vi na net tudo antes das ultimas 2 dimensoes que coloquei é rodado em paralelo)
        # (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        Q_matrix = Q_matrix.transpose(1,2)
        K_matrix = K_matrix.transpose(1,2)
        V_matrix = V_matrix.transpose(1,2)


        # Mesma ideia de multiplicar Q * K^T, mas como agr a dimensão da minha matriz ta tokens x dimensão_cortada_quant_cabecas é isso que uso e não d_in
        # tokens (palavras) x quantas colunas tenho representando esse "embedding" vetor essa ideia
        attention_scores_omegazinho = Q_matrix @ K_matrix.transpose(2, 3)

        mascara = self.mask
        attention_scores_omegazinho.masked_fill_(mascara.bool()[:num_tokens,:num_tokens], -torch.inf)


        # aqui vai ter que ser head_dim ao invés de d_out (é a msm ideia, mas como dividi em cabeças o numero de colunas ficou head_dim e não d_out)
        attention_weights = torch.softmax(attention_scores_omegazinho/self.head_dim**0.5, dim=-1) #é aquele trem de dividir por raiz da dimensão. Como quero que a soma de cada elemento numa linha (logo percorro colunas de uma linha) totalizem 1 uso "dim=-1" para isso.
        # attention_weights = torch.softmax(attention_scores_omegazinho/self.d_out**0.5, dim=-1)
        # o livro usa keys.shape[-1], mas pelo que entendi ao estudar é a mesma coisa que o d_out então da na mesma.

        attention_weights = self.dropout(attention_weights)

        vetor_contexto = attention_weights @ V_matrix

        # volta para formato de antes:
        # (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        vetor_contexto = vetor_contexto.transpose(1,2)

        # torno contínuo por exigência do .view() e depois eu agrupo tudo novamente na dimensão do d_out que é o que sairia
        vetor_contexto = vetor_contexto.contiguous().view(batch, num_tokens, self.d_out)

        vetor_contexto = self.Wout_proj(vetor_contexto) # Opicional pelo exemplo, mas lembro de na aula o professor usar. Então, vou usar professor

        return vetor_contexto, attention_weights


In [57]:
# CARREGUE O DATA LOADER COM O TEXTO ACIMA
# PASSE O PRIMEIRO BATCH PELA CAMADA DE EMBEDDING
# PASSE O EMBEDDING PELA MULTI HEAD ATTENTION
# IMPRIMA O SHAPE DA SAÍDA

max_length = 4
batch_size = 8
stride = 4
context_length = 4
num_heads = 2
d_in = emb_dim
d_out = emb_dim

data_loader = criar_data_loader(time_machine_text, batch_size=batch_size, tamanho_janela=max_length, stride=stride)


# pego o primeiro
input_batch, target_batch = next(iter(data_loader))

# print(input_batch.shape[1])

# Converte os IDs dos tokens em embeddings usando a camada do professor:
token_embeddings = token_embedding_layer(input_batch)

# Converte as posições [0, 1, 2, 3] usando a camada do professor:
pos_embeddings = pos_embedding_layer(torch.arange(input_batch.shape[1]))

# Soma os dois:
input_embeddings = token_embeddings + pos_embeddings # tenho o embedding e a ideia de posição. sentido+posicao


mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=num_heads)

context_vecs, _ = mha(input_embeddings)

# print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)



context_vecs.shape: torch.Size([8, 4, 256])
